In [ ]:
# Copyright 2026 AIT Austrian Institute of Technology GmbH
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# SODA-CitrON: Static Object Data Association by Clustering Multi-Modal Sensor Detections Online - Example Notebook

## Object generation

In [1]:
from scipy.stats import uniform
from shapely.geometry import Point
import geopandas as gpd

import uuid

# --------------------
# ROI parameters
# --------------------
ROI_MIN = 0.0
ROI_MAX = 150.0
ROI_AREA = ROI_MAX**2

# --------------------
# Objects (ground truth)
# --------------------
ground_truth_configs = [
    {
        "type": "A",
        "n_objects": 25,
        "detection_radius": 0.3,
    },
    {
        "type": "B",
        "n_objects": 25,
        "detection_radius": 0.2,
    },
    {
        "type": "C",
        "n_objects": 25,
        "detection_radius": 0.25,
    },
    {
        "type": "D",
        "n_objects": 25,
        "detection_radius": 0.45,
    },
]

ground_truth_objects = []

for gt_config in ground_truth_configs:
    for _ in range(gt_config["n_objects"]):
        position = uniform.rvs(ROI_MIN, ROI_MAX, 2)
        ground_truth_objects.append(
            {
                "id": uuid.uuid4(),
                "type": gt_config["type"],
                "detection_radius": gt_config["detection_radius"],
                "geometry": Point(*(position.tolist())), 
            }
        )

ground_truth_gdf = gpd.GeoDataFrame(ground_truth_objects)
ground_truth_gdf["x"] = ground_truth_gdf.geometry.x
ground_truth_gdf["y"] = ground_truth_gdf.geometry.y
ground_truth_gdf.head()

,id,type,detection_radius,geometry,x,y
0,45405564-07c0-44b3-ae81-09a9e91c094f,A,0.3,POINT (23.204 75.565),23.204379,75.564953
1,b26bf2c5-7d2b-4de2-b6ae-4eb55f8aa481,A,0.3,POINT (134.09 9.094),134.089735,9.094181
2,7bd0cff6-f23c-43b7-96f8-977d91346975,A,0.3,POINT (137.063 0.525),137.062804,0.524703
3,cbe4a40b-4cd0-4745-9bc4-deef0e156189,A,0.3,POINT (130.351 83.385),130.350668,83.384948
4,038c23d9-77f6-4e1e-b94b-81f3aff7abe0,A,0.3,POINT (95.763 141.474),95.763486,141.474495


In [2]:
import plotly.express as px

ground_truth_gdf["id"] = ground_truth_gdf["id"].astype(str)

fig = px.scatter(
    ground_truth_gdf,
    x="x",
    y="y",
    color="type",
    symbol_sequence=["x"],
    hover_data={"id": True, "type": True},
    title="Ground truth objects",
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)  # keep aspect ratio 1:1
fig.show()

## Sensor detection generation

In [51]:
import numpy as np
from scipy.stats import bernoulli, beta, norm, rv_discrete

class normal_discrete():
    def __init__(self, mean=0.0, std=1.0):
        # compute support before constructing rv_discrete
        k = np.arange(0, mean*(std+1) + 1)

        # compute probabilities
        p = norm.pdf(k, loc=mean, scale=std)
        p = p / p.sum()

        # Construct the actual scipy rv_discrete instance
        self.dist = rv_discrete(values=(k, p))

    def rvs(self):
        return self.dist.rvs()

# --------------------
# Sensor parameters
# --------------------

# define the sensor variances (95% radius^2/5.991)
sensor1_var = 0.3**2/5.991
sensor2_var = 1.0**2/5.991
sensor3_var = 0.7**2/5.991
sensor4_var = 0.7**2/5.991
sensor5_var = 1.5**2/5.991

sensors = [
    {
        "name": "SENSOR1",
        "detection_probabilities": {
            "A": 0.6,
            "B": 0.8,
            "C": 0.9,
            "D": 0.8,
        },
        "detections_per_object": {
            "A": bernoulli(p=1.0),
            "B": bernoulli(p=1.0),
            "C": bernoulli(p=1.0),
            "D": bernoulli(p=1.0),
        },
        "covariance": np.diag([sensor1_var, sensor1_var]),
        "detection_confidence": rv_discrete(values=([0.5, 0.75, 1.0], [0.25, 0.25, 0.50])),
        "clutter_confidence": rv_discrete(values=([0.5, 0.75, 1.0], [0.25, 0.25, 0.50])),
        "false_alarm_rate": 0.0005, # expected clutter points / m^2
    },
    {
        "name": "SENSOR2",
        "detection_probabilities": {
            "A": 0.8,
            "B": 0.0,
            "C": 0.6,
            "D": 0.6,
        },
        "detections_per_object": {
            "A": normal_discrete(mean=3, std=1),
            "B": normal_discrete(mean=3, std=1),
            "C": normal_discrete(mean=3, std=1),
            "D": normal_discrete(mean=3, std=1),
        },
        "covariance": np.diag([sensor2_var, sensor2_var]),
        "detection_confidence": beta(8, 2.5),
        "clutter_confidence": beta(8, 8),
        "false_alarm_rate": 0.02, # expected clutter points / m^2
    },
    {
        "name": "SENSOR3",
        "detection_probabilities": {
            "A": 0.0,
            "B": 0.85,
            "C": 0.6,
            "D": 0.6,
        },
        "detections_per_object": {
            "A": bernoulli(p=1.0),
            "B": bernoulli(p=1.0),
            "C": bernoulli(p=1.0),
            "D": bernoulli(p=1.0),
        },
        "covariance": np.diag([sensor3_var, sensor3_var]),
        "detection_confidence": beta(8, 2.5),
        "clutter_confidence": beta(8, 8),
        "false_alarm_rate": 0.01, # expected clutter points / m^2
    },
    {
        "name": "SENSOR4",
        "detection_probabilities": {
            "A": 0.7,
            "B": 0.7,
            "C": 0.7,
            "D": 0.7,
        },
        "detections_per_object": {
            "A": bernoulli(p=1.0),
            "B": bernoulli(p=1.0),
            "C": bernoulli(p=1.0),
            "D": bernoulli(p=1.0),
        },
        "covariance": np.diag([sensor4_var, sensor4_var]),
        "detection_confidence": beta(8, 2.5),
        "clutter_confidence": beta(8, 8),
        "false_alarm_rate": 0.01, # expected clutter points / m^2
    },
    {
        "name": "SENSOR5",
        "detection_probabilities": {
            "A": 0.8,
            "B": 0.3,
            "C": 0.7,
            "D": 0.7,
        },
        "detections_per_object": {
            "A": normal_discrete(mean=2, std=1),
            "B": normal_discrete(mean=2, std=1),
            "C": normal_discrete(mean=2, std=1),
            "D": normal_discrete(mean=2, std=1),
        },
        "covariance": np.diag([sensor5_var, sensor5_var]),
        "detection_confidence": beta(8, 2.5),
        "clutter_confidence": beta(8, 8),
        "false_alarm_rate": 0.02, # expected clutter points / m^2
    },
]

In [52]:
from ulid import ULID
from scipy.stats import multivariate_normal, poisson

# --------------------
# Create detections and clutter
# --------------------

object_positions = ground_truth_gdf[["x", "y"]].to_numpy()

clutter_rejection_radius = np.sqrt(0.3) * 3

true_detections = []
clutter_detections = []

for sensor in sensors:
    uncertainty_halo = np.sqrt(sensor["covariance"][0,0]*6) # radius contains 95% of detections
    
    for _, object in ground_truth_gdf.iterrows():
        P_D = sensor["detection_probabilities"][object.type]
        n_detections = sensor["detections_per_object"][object.type].rvs()
        if uniform.rvs() < P_D:
            for _ in range(n_detections):
                confidence = sensor["detection_confidence"].rvs()
                position = multivariate_normal.rvs(mean=object.geometry.coords[0], cov=sensor["covariance"])
                detection = {
                    "id": ULID(),
                    "sensor": sensor["name"],
                    "confidence": confidence,
                    "covariance": sensor["covariance"],
                    "geometry": Point(*(position.tolist())),
                    "uncertainty_halo": uncertainty_halo,
                    "ground_truth": object.id,
                }
                true_detections.append(detection)

    n_clutter = poisson.rvs(sensor["false_alarm_rate"]*ROI_AREA)
    for _ in range(n_clutter):
        confidence = sensor["clutter_confidence"].rvs()
        clutter_position = uniform.rvs(ROI_MIN, ROI_MAX, 2)
        while np.any(np.linalg.norm(object_positions-clutter_position, axis=1) < clutter_rejection_radius): # reject any clutter that could be a true detection
            clutter_position = uniform.rvs(ROI_MIN, ROI_MAX, 2)
        clutter = {
            "id": ULID(),
            "sensor": sensor["name"],
            "confidence": confidence,
            "covariance": sensor["covariance"],
            "geometry": Point(*(clutter_position.tolist())),
            "uncertainty_halo": uncertainty_halo,
            "ground_truth": "clutter",
        }
        clutter_detections.append(clutter)

all_detections = true_detections + clutter_detections

sensor_detections_gdf = gpd.GeoDataFrame(all_detections)
sensor_detections_gdf["x"] = sensor_detections_gdf.geometry.x
sensor_detections_gdf["y"] = sensor_detections_gdf.geometry.y
sensor_detections_gdf.head()

,id,sensor,confidence,covariance,geometry,uncertainty_halo,ground_truth,x,y
0,01KT6THHQGWSK8CGEFMKTJTRMN,SENSOR1,0.50,"[[0.015022533800701052, 0.0], [0.0, 0.01502253...",POINT (23.223 75.748),0.300225,45405564-07c0-44b3-ae81-09a9e91c094f,23.222883,75.748335
1,01KT6THHQKPV1Y6XYBHA99GWNN,SENSOR1,0.75,"[[0.015022533800701052, 0.0], [0.0, 0.01502253...",POINT (133.974 9.194),0.300225,b26bf2c5-7d2b-4de2-b6ae-4eb55f8aa481,133.974459,9.193617
2,01KT6THHQNFNYWKYCPZ0SHEBX2,SENSOR1,1.00,"[[0.015022533800701052, 0.0], [0.0, 0.01502253...",POINT (130.161 83.535),0.300225,cbe4a40b-4cd0-4745-9bc4-deef0e156189,130.161111,83.535183
3,01KT6THHQRDRG9K5WZ30E0VBAC,SENSOR1,1.00,"[[0.015022533800701052, 0.0], [0.0, 0.01502253...",POINT (84.875 72.658),0.300225,301c6443-657e-4740-b383-5124ff8953b3,84.875053,72.658225
4,01KT6THHQTKT72B3H5B6ED0R3K,SENSOR1,1.00,"[[0.015022533800701052, 0.0], [0.0, 0.01502253...",POINT (114.264 133.633),0.300225,d8854bbf-d7bb-4fef-9acc-e80c29bf3607,114.264367,133.633039


In [53]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ground_truth_gdf["x"],
    y=ground_truth_gdf["y"],
    mode="markers",
    marker=dict(symbol="x", color="green", size=10),
    name="ground truth",
    customdata=ground_truth_gdf[["id", "type"]],
    hovertemplate="id: %{customdata[0]}<br>type: %{customdata[1]}<extra></extra>",
))

sensor_detections_gdf["id"] = sensor_detections_gdf["id"].astype(str)

for sensor, group in sensor_detections_gdf.groupby("sensor"):
    fig.add_trace(go.Scatter(
        x=group["x"],
        y=group["y"],
        mode="markers",
        marker=dict(symbol="circle"),
        name=str(sensor),
        customdata=group[["id", "confidence", "ground_truth"]],
        hovertemplate="id: %{customdata[0]}<br>confidence: %{customdata[1]}<br>ground truth: %{customdata[2]}<extra></extra>",
    ))

fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(title="Sensor detections")
fig.show()

## Run SODA-CitrON

In [57]:
from soda_citron import SodaCitron

from functools import partial

def confidence2weight_exp(confidence: float, max_weight: float=1, scaling: float=1):
    return (np.exp(scaling*confidence)-1)/(np.exp(scaling)-1)*max_weight

confidence2weight = partial(confidence2weight_exp, max_weight=10, scaling=6)

soda_citron = SodaCitron(
    clustering_threshold=1.1,
    intersection_factor=0.3,
    minimum_weight=4.0,
)

In [58]:
for _, detection in sensor_detections_gdf.sample(frac=1).reset_index(drop=True).iterrows(): # randomize sensor detection sequence
    position = np.array(detection.geometry.coords[0])
    confidence = detection.confidence
    cov = np.array(detection.covariance)
    id = ULID.from_str(detection.id)
    weight = confidence2weight(confidence)

    soda_citron.learn_one(position, confidence, cov, w=weight, id=id)

# retrieve results
soda_citron_estimations = []
for _, c in soda_citron.clusters.items():
    soda_citron_estimations.append(
        {
            "id": c.id.hex,
            "confidence": c.conf,
            "covariance": c.cov,
            "weight": c.weight,
            "geometry": Point(*c.center.tolist()),
            "x": c.center[0], 
            "y": c.center[1],
        }
    )

soda_citron_estimations_gdf = gpd.GeoDataFrame(soda_citron_estimations)
soda_citron_estimations_gdf["x"] = soda_citron_estimations_gdf.geometry.x
soda_citron_estimations_gdf["y"] = soda_citron_estimations_gdf.geometry.y
soda_citron_estimations_gdf.head()

,id,confidence,covariance,weight,geometry,x,y
0,019e8da96a80889c7fce62f4de61fda7,0.999999,"[[0.01995554908456471, 0.0], [0.0, 0.019955549...",32.315646,POINT (87.998 100.012),87.998129,100.012486
1,019e8da96a81fa27d15d35ce57f995f1,1.000000,"[[0.01067436421453526, 0.0], [0.0, 0.010674364...",21.861327,POINT (133.786 31.051),133.785832,31.051076
2,019e8da96a83a03ffe8a6573e0f36de4,0.997003,"[[0.04890983652421081, 0.0], [0.0, 0.048909836...",14.499237,POINT (50.891 124.796),50.890613,124.795534
3,019e8da96a8ddd6ca7f0558b899e2e44,0.964290,"[[0.011304785850823262, 0.0], [0.0, 0.01130478...",6.718086,POINT (20.611 108.25),20.610676,108.250033
4,019e8da96a8f18384bb43d7c7e4093bc,0.988679,"[[0.037837411284278075, 0.0], [0.0, 0.03783741...",11.949536,POINT (127.591 111.722),127.590564,111.721563


In [59]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=ground_truth_gdf["x"],
    y=ground_truth_gdf["y"],
    mode="markers",
    marker=dict(symbol="x", color="green", size=10),
    name="ground truth",
    customdata=ground_truth_gdf[["id", "type"]],
    hovertemplate="id: %{customdata[0]}<br>type: %{customdata[1]}<extra></extra>",
))

soda_citron_estimations_gdf["id"] = soda_citron_estimations_gdf["id"].astype(str)

fig.add_trace(go.Scatter(
    x=soda_citron_estimations_gdf["x"],
    y=soda_citron_estimations_gdf["y"],
    mode="markers",
    marker=dict(symbol="circle", color="red", size=10),
    name="SODA-CitrON estimations",
    customdata=soda_citron_estimations_gdf[["id", "confidence"]],
    hovertemplate="id: %{customdata[0]}<br>confidence: %{customdata[1]}<extra></extra>",
))

fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(title="SODA-CitrON estimations")
fig.show()